<a href="https://colab.research.google.com/github/Narenkumar19/iot_project/blob/main/IoT_Based_Intelligent_Battery_and_Device_Health_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install psutil plotly

In [2]:
import psutil
import time
import platform
import socket
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import clear_output, display
from datetime import datetime

# ============================================================
# SMART DEVICE BMS
# Software-Based Device Health Monitoring System
# ============================================================

history = []

MAX_HISTORY = 50


# ============================================================
# GET CPU TEMPERATURE
# ============================================================

def get_temperature():

    try:

        temperatures = psutil.sensors_temperatures()

        if temperatures:

            for name, entries in temperatures.items():

                for entry in entries:

                    if entry.current is not None:
                        return float(entry.current)

    except:
        pass

    # Temperature unavailable
    return np.nan


# ============================================================
# GET BATTERY INFORMATION
# ============================================================

def get_battery():

    try:

        battery = psutil.sensors_battery()

        if battery is not None:

            return battery.percent, battery.power_plugged

    except:
        pass

    return np.nan, None


# ============================================================
# GET DEVICE DATA
# ============================================================

def get_device_data():

    # CPU
    cpu_usage = psutil.cpu_percent(interval=1)

    # RAM
    ram_usage = psutil.virtual_memory().percent

    # Disk
    disk_usage = psutil.disk_usage('/').percent

    # Temperature
    temperature = get_temperature()

    # Battery
    battery_level, plugged = get_battery()

    # Network
    network = psutil.net_io_counters()

    bytes_sent = network.bytes_sent / (1024 * 1024)
    bytes_received = network.bytes_recv / (1024 * 1024)

    # Uptime
    boot_time = psutil.boot_time()

    uptime_seconds = time.time() - boot_time

    uptime_hours = uptime_seconds / 3600

    return {
        "Time": datetime.now(),
        "CPU": cpu_usage,
        "RAM": ram_usage,
        "Disk": disk_usage,
        "Temperature": temperature,
        "Battery": battery_level,
        "Plugged": plugged,
        "Sent_MB": bytes_sent,
        "Received_MB": bytes_received,
        "Uptime": uptime_hours
    }


# ============================================================
# DEVICE HEALTH SCORE
# ============================================================

def calculate_health(data):

    score = 100

    # CPU penalty
    if data["CPU"] > 90:
        score -= 25

    elif data["CPU"] > 75:
        score -= 10

    # RAM penalty
    if data["RAM"] > 90:
        score -= 20

    elif data["RAM"] > 75:
        score -= 10

    # Disk penalty
    if data["Disk"] > 95:
        score -= 20

    elif data["Disk"] > 85:
        score -= 10

    # Temperature penalty
    if not np.isnan(data["Temperature"]):

        if data["Temperature"] > 85:
            score -= 30

        elif data["Temperature"] > 70:
            score -= 15

    # Battery penalty
    if not np.isnan(data["Battery"]):

        if data["Battery"] < 10:
            score -= 20

        elif data["Battery"] < 20:
            score -= 10

    score = max(0, min(100, score))

    # Status
    if score >= 80:
        status = "NORMAL"

    elif score >= 60:
        status = "WARNING"

    else:
        status = "CRITICAL"

    return score, status


# ============================================================
# ALERT GENERATOR
# ============================================================

def generate_alerts(data):

    alerts = []

    if data["CPU"] > 90:
        alerts.append("HIGH CPU USAGE")

    if data["RAM"] > 90:
        alerts.append("HIGH RAM USAGE")

    if data["Disk"] > 95:
        alerts.append("DISK ALMOST FULL")

    if not np.isnan(data["Temperature"]):

        if data["Temperature"] > 85:
            alerts.append("CRITICAL TEMPERATURE")

        elif data["Temperature"] > 70:
            alerts.append("HIGH TEMPERATURE")

    if not np.isnan(data["Battery"]):

        if data["Battery"] < 10:
            alerts.append("CRITICAL BATTERY")

        elif data["Battery"] < 20:
            alerts.append("LOW BATTERY")

    return alerts


# ============================================================
# CREATE GAUGE
# ============================================================

def create_gauge(value, title, minimum=0, maximum=100, suffix="%"):

    if np.isnan(value):
        value = 0
        number_text = "N/A"

    else:
        number_text = f"{value:.1f}{suffix}"

    fig = go.Figure(
        go.Indicator(
            mode="gauge+number",
            value=value,
            title={"text": title},
            number={
                "suffix": suffix,
                "valueformat": ".1f"
            },
            gauge={
                "axis": {
                    "range": [minimum, maximum]
                }
            }
        )
    )

    fig.update_layout(
        height=250,
        margin=dict(
            l=20,
            r=20,
            t=60,
            b=20
        )
    )

    return fig


# ============================================================
# DASHBOARD
# ============================================================

def update_dashboard():

    global history

    data = get_device_data()

    history.append(data)

    history = history[-MAX_HISTORY:]

    df = pd.DataFrame(history)

    health_score, status = calculate_health(data)

    alerts = generate_alerts(data)

    clear_output(wait=True)

    # ========================================================
    # HEADER
    # ========================================================

    print("=" * 75)
    print("          SMART IoT DEVICE BATTERY MANAGEMENT SYSTEM")
    print("                 DEVICE HEALTH DASHBOARD")
    print("=" * 75)

    print()

    print("DEVICE INFORMATION")
    print("-" * 75)

    print("Operating System :", platform.system())
    print("Platform         :", platform.platform())
    print("Hostname         :", socket.gethostname())

    print()

    # ========================================================
    # FIVE PARAMETERS
    # ========================================================

    print("MONITORED PARAMETERS")
    print("-" * 75)

    print(f"CPU Usage        : {data['CPU']:.2f} %")
    print(f"RAM Usage        : {data['RAM']:.2f} %")
    print(f"Disk Usage       : {data['Disk']:.2f} %")

    if np.isnan(data["Temperature"]):
        print("CPU Temperature  : Not available")
    else:
        print(f"CPU Temperature  : {data['Temperature']:.2f} °C")

    if np.isnan(data["Battery"]):
        print("Battery Level    : Not available")
    else:

        power_status = "Plugged In" if data["Plugged"] else "Battery"

        print(
            f"Battery Level    : "
            f"{data['Battery']:.2f} % "
            f"({power_status})"
        )

    print()

    # ========================================================
    # HEALTH
    # ========================================================

    print("DEVICE HEALTH")
    print("-" * 75)

    print(f"Health Score     : {health_score:.1f} / 100")
    print(f"Device Status    : {status}")

    print(f"System Uptime    : {data['Uptime']:.2f} hours")

    print()

    # ========================================================
    # ALERTS
    # ========================================================

    print("BMS ALERTS")
    print("-" * 75)

    if len(alerts) == 0:

        print("✓ No abnormal conditions detected")

    else:

        for alert in alerts:
            print("⚠", alert)

    print()

    # ========================================================
    # GAUGES
    # ========================================================

    print("REAL-TIME MONITORING")
    print("-" * 75)

    display(
        create_gauge(
            data["CPU"],
            "CPU Usage"
        )
    )

    display(
        create_gauge(
            data["RAM"],
            "RAM Usage"
        )
    )

    display(
        create_gauge(
            data["Disk"],
            "Disk Usage"
        )
    )

    if not np.isnan(data["Temperature"]):

        display(
            create_gauge(
                data["Temperature"],
                "CPU Temperature",
                0,
                100,
                " °C"
            )
        )

    if not np.isnan(data["Battery"]):

        display(
            create_gauge(
                data["Battery"],
                "Battery Level"
            )
        )

    # ========================================================
    # HISTORICAL GRAPH
    # ========================================================

    if len(df) > 1:

        fig = make_subplots(
            rows=5,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.04,
            subplot_titles=[
                "CPU Usage (%)",
                "RAM Usage (%)",
                "Disk Usage (%)",
                "Temperature (°C)",
                "Battery Level (%)"
            ]
        )

        # CPU
        fig.add_trace(
            go.Scatter(
                x=df["Time"],
                y=df["CPU"],
                mode="lines+markers",
                name="CPU"
            ),
            row=1,
            col=1
        )

        # RAM
        fig.add_trace(
            go.Scatter(
                x=df["Time"],
                y=df["RAM"],
                mode="lines+markers",
                name="RAM"
            ),
            row=2,
            col=1
        )

        # Disk
        fig.add_trace(
            go.Scatter(
                x=df["Time"],
                y=df["Disk"],
                mode="lines+markers",
                name="Disk"
            ),
            row=3,
            col=1
        )

        # Temperature
        fig.add_trace(
            go.Scatter(
                x=df["Time"],
                y=df["Temperature"],
                mode="lines+markers",
                name="Temperature"
            ),
            row=4,
            col=1
        )

        # Battery
        fig.add_trace(
            go.Scatter(
                x=df["Time"],
                y=df["Battery"],
                mode="lines+markers",
                name="Battery"
            ),
            row=5,
            col=1
        )

        fig.update_layout(
            height=1200,
            title="Device BMS Historical Monitoring",
            showlegend=False
        )

        display(fig)


# ============================================================
# START DASHBOARD
# ============================================================

update_dashboard()

          SMART IoT DEVICE BATTERY MANAGEMENT SYSTEM
                 DEVICE HEALTH DASHBOARD

DEVICE INFORMATION
---------------------------------------------------------------------------
Operating System : Linux
Platform         : Linux-6.6.122+-x86_64-with-glibc2.35
Hostname         : 816864e1ccdc

MONITORED PARAMETERS
---------------------------------------------------------------------------
CPU Usage        : 3.00 %
RAM Usage        : 8.90 %
Disk Usage       : 18.90 %
CPU Temperature  : Not available
Battery Level    : Not available

DEVICE HEALTH
---------------------------------------------------------------------------
Health Score     : 100.0 / 100
Device Status    : NORMAL
System Uptime    : 0.38 hours

BMS ALERTS
---------------------------------------------------------------------------
✓ No abnormal conditions detected

REAL-TIME MONITORING
---------------------------------------------------------------------------


In [3]:
# ============================================================
# LIVE DEVICE BMS MONITORING
# ============================================================

print("Starting Smart Device BMS...")
print("Refresh interval: 5 seconds")
print("Press STOP/INTERRUPT to stop monitoring.")

try:

    while True:

        update_dashboard()

        time.sleep(5)

except KeyboardInterrupt:

    print("\nDevice BMS monitoring stopped.")

          SMART IoT DEVICE BATTERY MANAGEMENT SYSTEM
                 DEVICE HEALTH DASHBOARD

DEVICE INFORMATION
---------------------------------------------------------------------------
Operating System : Linux
Platform         : Linux-6.6.122+-x86_64-with-glibc2.35
Hostname         : 816864e1ccdc

MONITORED PARAMETERS
---------------------------------------------------------------------------
CPU Usage        : 1.50 %
RAM Usage        : 9.40 %
Disk Usage       : 18.90 %
CPU Temperature  : Not available
Battery Level    : Not available

DEVICE HEALTH
---------------------------------------------------------------------------
Health Score     : 100.0 / 100
Device Status    : NORMAL
System Uptime    : 0.40 hours

BMS ALERTS
---------------------------------------------------------------------------
✓ No abnormal conditions detected

REAL-TIME MONITORING
---------------------------------------------------------------------------



Device BMS monitoring stopped.
